In [14]:
import duckdb
import geoarrow.pyarrow as ga
from lonboard import Map, PolygonLayer
from lonboard.basemap import MaplibreBasemap, CartoStyle

In [10]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

In [11]:
arrow_table = con.execute("""
    SELECT ST_AsWKB(geom) AS geometry,
           bin, height_roof, construction_year
    FROM ST_Read('output.geojson')
""").arrow().read_all()

# Wrap as GeoArrow WKB so Lonboard recognizes geometry
geom_col = ga.as_wkb(arrow_table["geometry"])
table = arrow_table.set_column(
    arrow_table.schema.get_field_index("geometry"),
    "geometry", geom_col
)

In [18]:
layer = PolygonLayer(
    table=table,
    get_fill_color=[0, 100, 200, 120],   # semi-transparent blue
    get_line_color=[255, 255, 255],
    get_line_width=1,
    line_width_units="pixels",
    pickable=True,
)

m = Map(
    layers=[layer],
    basemap=MaplibreBasemap(style=CartoStyle.Positron),
    view_state={
        "longitude": -74.006,
        "latitude": 40.712,
        "zoom": 14,
        "pitch": 0,
        "bearing": 0,
    },
)
m.to_html("map.html")